## Preliminaries

Import the necessary libraries and specify the data folders.

In [ ]:
import os
# --- CUDA / libdevice paths (adjust to your local environment) ---
# os.environ["CUDA_PATH"] = r"path/to/your/cuda"
# os.environ["NVVM_LIBDEVICE_PATH"] = r"path/to/your/cuda/nvvm/libdevice"

import tensorflow as tf
tf.config.run_functions_eagerly(False)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print("Could not set memory growth:", e)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import deepxde as dde
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
import pickle
import glob
import shutil

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
DATA_PATH = "../Data/"
INPUT_MODEL_PATH = DATA_PATH + "processed_data/"
MODEL_SAVE_PATH = DATA_PATH + "trained_models/"
RESULTS_PATH = DATA_PATH + "model_results/"

In [ ]:
from style_jcp import load_world
world = load_world()

In [ ]:
with open("functions_training_model.py", 'r') as file:
    content = file.read()

# Execute the content of the .py file
exec(content)

In [ ]:
def denormalize_predictions(normalized_values, lat_values, norm_params):
    global_mean = norm_params['global_mean']
    global_std = norm_params['global_std']
    denorm = normalized_values * global_std + global_mean

    # Apply bias corrections if present
    if 'bias_corrections' in norm_params and norm_params['bias_corrections']:
        lat_bands = norm_params['lat_bands']
        lat_band_indices = pd.cut(lat_values, bins=lat_bands, include_lowest=True)

        for band_str, correction_info in norm_params['bias_corrections'].items():
            mask = (lat_band_indices.astype(str) == band_str)
            if mask.any():
                denorm[mask] += correction_info['correction'] * global_std

    return denorm


In [ ]:
def calculate_save_df_universal(model, df_to_predict, norm_params, path, filename):
    import numpy as np

    preds = model.predict(df_to_predict[['lon', 'lat']].values / 90)

    U_pred_norm = preds[:, 0]
    D_raw = preds[:, 1]

    D_pred = np.logaddexp(0, D_raw)

    U_pred_denorm = denormalize_predictions(
        U_pred_norm,
        df_to_predict['lat'].values,
        norm_params
    )

    df_to_predict['PINN_log_dep'] = U_pred_denorm
    df_to_predict['PINN_Diffusivity'] = D_pred

    with open(path + filename, 'w') as f:
        df_to_predict.to_csv(f, index=False)

    print(f"✓ Saved: {filename}")
    print(f"  Deposition Range: [{U_pred_denorm.min():.3f}, {U_pred_denorm.max():.3f}]")
    print(f"  Diffusivity Range: [{D_pred.min():.4f}, {D_pred.max():.4f}]")

In [ ]:
def calculate_save_df_constD(model, df_to_predict, norm_params, path, filename):
    """单输出 constD 模型的预测保存（仅 u，无 D 场）"""
    preds = model.predict(df_to_predict[['lon', 'lat']].values / 90)
    U_pred_norm = preds[:, 0]

    U_pred_denorm = denormalize_predictions(
        U_pred_norm,
        df_to_predict['lat'].values,
        norm_params
    )

    df_to_predict['PINN_log_dep'] = U_pred_denorm

    with open(path + filename, 'w') as f:
        df_to_predict.to_csv(f, index=False)

    print(f"✓ [constD] Saved: {filename}")
    print(f"  Deposition Range: [{U_pred_denorm.min():.3f}, {U_pred_denorm.max():.3f}]")

In [ ]:
df_simulated_Holocene_raw = pd.read_csv(INPUT_MODEL_PATH + "df_simulation_Holocene.csv")
df_simulated_LGM_raw = pd.read_csv(INPUT_MODEL_PATH + "df_simulation_LGM.csv")
df_hol_safe = df_simulated_Holocene_raw[(df_simulated_Holocene_raw['lat'] >= -80) & (df_simulated_Holocene_raw['lat'] <= 80)]
df_lgm_safe = df_simulated_LGM_raw[(df_simulated_LGM_raw['lat'] >= -80) & (df_simulated_LGM_raw['lat'] <= 80)]

if len(df_hol_safe) > 2000:
    df_simulated_Holocene = df_hol_safe.sample(n=2000, random_state=42).copy()
else:
    df_simulated_Holocene = df_hol_safe.copy()

if len(df_lgm_safe) > 30:
    df_simulated_LGM = df_lgm_safe.sample(n=30, random_state=42).copy()
else:
    df_simulated_LGM = df_lgm_safe.copy()


In [ ]:
df_global_grid = pd.read_csv(INPUT_MODEL_PATH + "df_global_grid.csv")

In [ ]:
df_wind = pd.read_csv(INPUT_MODEL_PATH + "df_wind.csv", usecols=['wind', 'latitude'])

In [ ]:
from functions_training_model import wind_tf_interp

latitude_wind, mean_wind = df_wind['latitude'].values/90, df_wind['wind'].values/df_wind['wind'].max()

@tf.function
def wind_latitude(latitude):
    with tf.device("/GPU:0"):
        return wind_tf_interp(latitude, tf.convert_to_tensor(latitude_wind), tf.convert_to_tensor(mean_wind))

In [ ]:
def training_points(df):

    data_observ_points = dde.data.DataSet(
        X_train = df[['lon', 'lat']].values/90,
        y_train = df['log_dep_norm'].values.reshape(-1,1),
        X_test = df[['lon', 'lat']].values/90,
        y_test = df['log_dep_norm'].values.reshape(-1,1),
        standardize=False)

    observe_u  = dde.icbc.PointSetBC(data_observ_points.train_x,
        df['log_dep_norm'].values.reshape(-1,1), component=0)

    return data_observ_points, observe_u

In [ ]:
x_min, x_max = -2.0, 2.0
y_min, y_max = -0.89, 0.89

left_corner = np.array([x_min, y_min])
right_corner = np.array([x_max, y_max])
geometry_rectangle = dde.geometry.geometry_2d.Rectangle(left_corner, right_corner)

df_simulated_Holocene_2 = df_simulated_Holocene[(df_simulated_Holocene['lat'] >= -81) & (df_simulated_Holocene['lat'] <= 81)]
df_simulated_LGM_2 = df_simulated_LGM[(df_simulated_LGM['lat'] >= -81) & (df_simulated_LGM['lat'] <= 81)]

In [ ]:
def pde(x, y):
    u = y[:, 0:1]
    D_raw = y[:, 1:2]
    D = tf.math.softplus(D_raw)
    scale_1st = 2.0 / np.pi
    scale_2nd = (2.0 / np.pi) ** 2

    du_x_raw = dde.grad.jacobian(u, x, j=0)
    du_y_raw = dde.grad.jacobian(u, x, j=1)
    du_xx_raw = dde.grad.hessian(u, x, i=0, j=0)
    du_yy_raw = dde.grad.hessian(u, x, i=1, j=1)

    du_x = du_x_raw * scale_1st
    du_y = du_y_raw * scale_1st
    du_xx = du_xx_raw * scale_2nd
    du_yy = du_yy_raw * scale_2nd

    dD_x_raw = dde.grad.jacobian(D, x, j=0)
    dD_y_raw = dde.grad.jacobian(D, x, j=1)
    
    dD_x = dD_x_raw * scale_1st
    dD_y = dD_y_raw * scale_1st

    K = wind_latitude(x[:, 1:2])
    K = tf.cast(K, tf.float32)

    cos_theta = tf.cos(x[:, 1:2] * np.pi / 2)
    tan_theta = tf.tan(x[:, 1:2] * np.pi / 2)

    advection = -K * du_x * (1 / cos_theta)

    diffusion_main = D * ( (1/(cos_theta**2)) * du_xx + du_yy - tan_theta * du_y )

    diffusion_grad = (1/(cos_theta**2)) * dD_x * du_x + dD_y * du_y

    pde_residual = advection + diffusion_main + diffusion_grad

    grad_D_sq = (1/(cos_theta**2)) * dD_x**2 + dD_y**2

    return [pde_residual, grad_D_sq]

In [ ]:
def pde_constD(x, u):
    du_x_raw = dde.grad.jacobian(u, x, j=0)
    du_y_raw = dde.grad.jacobian(u, x, j=1)
    du_xx_raw = dde.grad.hessian(u, x, i=0, j=0)
    du_yy_raw = dde.grad.hessian(u, x, i=1, j=1)

    scale_1st = 2.0 / np.pi
    scale_2nd = (2.0 / np.pi) ** 2
    du_x = du_x_raw * scale_1st
    du_y = du_y_raw * scale_1st
    du_xx = du_xx_raw * scale_2nd
    du_yy = du_yy_raw * scale_2nd

    K = wind_latitude(x[:, 1:2])
    K = tf.cast(K, tf.float32)
    cos_theta = tf.cos(x[:, 1:2] * np.pi / 2)
    tan_theta = tf.tan(x[:, 1:2] * np.pi / 2)

    return (-K * du_x * (1 / cos_theta)
            + D_const * ((1 / (cos_theta**2)) * du_xx + du_yy - tan_theta * du_y))

In [ ]:
def space_boundary_north(x, on_boundary):
    return on_boundary and np.isclose(y_max, x[1])

def space_boundary_south(x, on_boundary):
    return on_boundary and np.isclose(y_min, x[1])

def periodic_boundary(x, domain):
    return domain and (np.isclose(x[0], x_min) or np.isclose(x[0], x_max))

In [ ]:
def train_process(data_observ_points, observe_u, bc_1, bc_2, model_name,
                  restore_path=None, epochs=150000, lr=0.00001, log_period=1000):

    save_dir = os.path.join(MODEL_SAVE_PATH, model_name)
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    ckpt_path = os.path.join(save_dir, f"{model_name}.ckpt")

    data = dde.data.PDE(
        geometry_rectangle,
        pde,
        [observe_u, periodic_condition, periodic_condition_derivative,
         periodic_D, periodic_D_derivative,
         bc_1, bc_2],
        num_domain=2592,
        num_boundary=216,
        anchors=data_observ_points.train_x,
        train_distribution='uniform'
    )

    neurons = 32
    layer = 5

    layer_size = [2] + [neurons] * layer + [2]
    activation = "selu"
    initializer = "Glorot normal"
    net = dde.maps.FNN(layer_size, activation, initializer)
    model = dde.Model(data, net)

    loss_weights = [1, 0.1, 10, 0.5, 0.5, 1, 1, 1, 1]

    dde.optimizers.set_LBFGS_options(maxcor=50, ftol=1e-20, maxiter=1e5)

    model.compile("adam", lr=lr, external_trainable_variables=[north_mean, south_mean],
                  loss_weights=loss_weights)

    checkpointer = dde.callbacks.ModelCheckpoint(
        ckpt_path, verbose=0, period=log_period, save_better_only=False
    )
    variable = dde.callbacks.VariableValue([north_mean, south_mean], period=log_period, filename=os.path.join(save_dir, "variables.dat"))

    if restore_path is not None:
        losshistory, train_state = model.train(epochs=epochs, callbacks=[variable, checkpointer], model_restore_path=restore_path)
    else:
        losshistory, train_state = model.train(epochs=epochs, callbacks=[variable, checkpointer])

    model.saver.save(model.sess, ckpt_path, global_step=model.train_state.step)
    dde.saveplot(losshistory, train_state, issave=True, isplot=False)

    target_loss_file = os.path.join(RESULTS_PATH, f"loss_history_{model_name}.dat")
    if os.path.exists("loss.dat"): shutil.move("loss.dat", target_loss_file)
    if os.path.exists("train.dat"): os.remove("train.dat")
    if os.path.exists("test.dat"): os.remove("test.dat")

    return model, variable.get_value(), train_state.best_step

In [ ]:
def train_process_constD(data_observ_points, observe_u, D_var, bc_1, bc_2, model_name,
                         restore_path=None, epochs=150000, lr=0.00001, log_period=1000):

    save_dir = os.path.join(MODEL_SAVE_PATH, model_name)
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    ckpt_path = os.path.join(save_dir, f"{model_name}.ckpt")

    data = dde.data.PDE(
        geometry_rectangle,
        pde_constD,
        [observe_u, periodic_condition, periodic_condition_derivative, bc_1, bc_2],
        num_domain=2592,
        num_boundary=216,
        anchors=data_observ_points.train_x,
        train_distribution='uniform'
    )

    neurons = 32
    layer = 5
    layer_size = [2] + [neurons] * layer + [1]
    activation = "selu"
    initializer = "Glorot normal"
    net = dde.maps.FNN(layer_size, activation, initializer)
    model = dde.Model(data, net)

    loss_weights = [1, 10, 0.5, 0.5, 1, 1]

    dde.optimizers.set_LBFGS_options(maxcor=50, ftol=1e-20, maxiter=1e5)

    model.compile("adam", lr=lr,
                  external_trainable_variables=[D_var, north_mean, south_mean],
                  loss_weights=loss_weights)

    checkpointer = dde.callbacks.ModelCheckpoint(
        ckpt_path, verbose=0, period=log_period, save_better_only=False)
    variable = dde.callbacks.VariableValue(
        [D_var, north_mean, south_mean], period=log_period,
        filename=os.path.join(save_dir, "variables.dat"))

    if restore_path is not None:
        losshistory, train_state = model.train(
            epochs=epochs, callbacks=[variable, checkpointer],
            model_restore_path=restore_path)
    else:
        losshistory, train_state = model.train(
            epochs=epochs, callbacks=[variable, checkpointer])

    model.saver.save(model.sess, ckpt_path, global_step=model.train_state.step)
    dde.saveplot(losshistory, train_state, issave=True, isplot=False)

    target_loss_file = os.path.join(RESULTS_PATH, f"loss_history_{model_name}.dat")
    if os.path.exists("loss.dat"): shutil.move("loss.dat", target_loss_file)
    if os.path.exists("train.dat"): os.remove("train.dat")
    if os.path.exists("test.dat"): os.remove("test.dat")

    return model, variable.get_value(), train_state.best_step

In [ ]:
df_LGM_full_test = pd.read_csv(INPUT_MODEL_PATH + "df_simulation_LGM.csv")
df_LGM_full_test = df_LGM_full_test[(df_LGM_full_test['lat'] >= -80) & (df_LGM_full_test['lat'] <= 80)]

tf.compat.v1.reset_default_graph()

model_name_hol = 'model_simulated_Holocene'

north_mean = dde.Variable(-1.0)
south_mean = dde.Variable(-2.0)

bc_1 = dde.DirichletBC(geometry_rectangle, lambda x: north_mean, space_boundary_north)
bc_2 = dde.DirichletBC(geometry_rectangle, lambda x: south_mean, space_boundary_south)

periodic_condition = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=0, component=0)
periodic_condition_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=1, component=0)

periodic_D = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=0, component=1)
periodic_D_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=1, component=1)

data_observ_points_Holocene, observe_u_Holocene = training_points(df_simulated_Holocene)

model_simulated_Holocene, params_hol, best_step_hol = train_process(
    data_observ_points_Holocene,
    observe_u_Holocene,
    bc_1, bc_2,
    model_name_hol,
    restore_path=None,
    epochs=150000, lr=0.00001
)

In [ ]:
with open(INPUT_MODEL_PATH + "norm_params_simulation_Holocene.pkl", 'rb') as f:
    norm_params_Holocene = pickle.load(f)

with open(INPUT_MODEL_PATH + "norm_params_simulation_LGM.pkl", 'rb') as f:
    norm_params_LGM = pickle.load(f)

In [ ]:
calculate_save_df_universal(
    model_simulated_Holocene,
    df_simulated_Holocene.copy(),
    norm_params_Holocene,
    RESULTS_PATH,
    "df_pinn_simulated_Holocene.csv"
)

In [ ]:
ckpt_dir = os.path.join(MODEL_SAVE_PATH, 'model_simulated_Holocene')
best_ckpt = os.path.join(ckpt_dir, f"model_simulated_Holocene.ckpt-{best_step_hol}.ckpt")
if os.path.exists(best_ckpt + '.index'):
    holocene_restore_path = best_ckpt
else:
    list_of_files = glob.glob(os.path.join(ckpt_dir, '*.index'))
    if list_of_files:
        latest_file = max(list_of_files, key=os.path.getctime)
        holocene_restore_path = latest_file.replace('.index', '')
    else:
        raise FileNotFoundError(f"❌ 未找到 Holocene 的 Checkpoint。请检查 Stage 1 是否成功运行。")

In [ ]:
tf.compat.v1.reset_default_graph()

model_name_lgm = 'model_simulated_LGM'

north_mean = dde.Variable(-1.0)
south_mean = dde.Variable(-2.0)

bc_1 = dde.DirichletBC(geometry_rectangle, lambda x: north_mean, space_boundary_north)
bc_2 = dde.DirichletBC(geometry_rectangle, lambda x: south_mean, space_boundary_south)

periodic_condition = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=0, component=0)
periodic_condition_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=1, component=0)

periodic_D = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=0, component=1)
periodic_D_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=1, component=1)

data_observ_points_LGM, observe_u_LGM = training_points(df_simulated_LGM)

model_simulated_LGM, params_lgm, _ = train_process(
    data_observ_points_LGM, observe_u_LGM, bc_1, bc_2,
    'model_simulated_LGM', restore_path=holocene_restore_path,
    epochs=150000, lr=1e-5
)

In [ ]:
calculate_save_df_universal(
    model_simulated_LGM,
    df_LGM_full_test.copy(),
    norm_params_LGM,
    RESULTS_PATH,
    "df_pinn_simulated_LGM.csv"
)

In [ ]:
tf.compat.v1.reset_default_graph()

model_name_base = 'model_simulated_LGM_Baseline'

north_mean = dde.Variable(-1.0)
south_mean = dde.Variable(-2.0)

bc_1 = dde.DirichletBC(geometry_rectangle, lambda x: north_mean, space_boundary_north)
bc_2 = dde.DirichletBC(geometry_rectangle, lambda x: south_mean, space_boundary_south)

periodic_condition = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=0, component=0)
periodic_condition_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=1, component=0)

periodic_D = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=0, component=1)
periodic_D_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0, on_boundary=periodic_boundary, derivative_order=1, component=1)

data_base, obs_u_base = training_points(df_simulated_LGM)

model_base, params_base, _ = train_process(
    data_base, obs_u_base, bc_1, bc_2,
    'model_simulated_LGM_Baseline', restore_path=None,
    epochs=150000, lr=0.00001
)

In [ ]:
print("\nEvaluating Baseline on FULL Data...")
calculate_save_df_universal(
    model_base,
    df_LGM_full_test.copy(),
    norm_params_LGM,
    RESULTS_PATH,
    "df_pinn_simulated_LGM_Baseline.csv"
)

In [ ]:
tf.compat.v1.reset_default_graph()

model_name_hol_constD = 'model_simulated_Holocene_constD'
D_const = dde.Variable(1.0)
north_mean = dde.Variable(-1.0)
south_mean = dde.Variable(-2.0)

bc_1 = dde.DirichletBC(geometry_rectangle, lambda x: north_mean, space_boundary_north)
bc_2 = dde.DirichletBC(geometry_rectangle, lambda x: south_mean, space_boundary_south)
periodic_condition = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0,
    on_boundary=periodic_boundary, derivative_order=0, component=0)
periodic_condition_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0,
    on_boundary=periodic_boundary, derivative_order=1, component=0)

data_observ_points_Holocene, observe_u_Holocene = training_points(df_simulated_Holocene)

model_hol_constD, params_hol_constD, best_step_hol_constD = train_process_constD(
    data_observ_points_Holocene,
    observe_u_Holocene,
    D_const, bc_1, bc_2,
    model_name_hol_constD,
    restore_path=None,
    epochs=150000, lr=1e-5
)

ckpt_dir_constD = os.path.join(MODEL_SAVE_PATH, model_name_hol_constD)
best_ckpt_constD = os.path.join(ckpt_dir_constD,
    f"{model_name_hol_constD}.ckpt-{best_step_hol_constD}.ckpt")

if os.path.exists(best_ckpt_constD + '.index'):
    holocene_constD_restore_path = best_ckpt_constD
else:
    list_of_files = glob.glob(os.path.join(ckpt_dir_constD, '*.index'))
    if list_of_files:
        latest_file = max(list_of_files, key=os.path.getctime)
        holocene_constD_restore_path = latest_file.replace('.index', '')
    else:
        raise FileNotFoundError(f"❌ 未找到 constD Holocene Checkpoint")

In [ ]:
tf.compat.v1.reset_default_graph()

model_name_lgm_constD = 'model_simulated_LGM_constD'
D_const = dde.Variable(1.0)
north_mean = dde.Variable(-1.0)
south_mean = dde.Variable(-2.0)

bc_1 = dde.DirichletBC(geometry_rectangle, lambda x: north_mean, space_boundary_north)
bc_2 = dde.DirichletBC(geometry_rectangle, lambda x: south_mean, space_boundary_south)
periodic_condition = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0,
    on_boundary=periodic_boundary, derivative_order=0, component=0)
periodic_condition_derivative = dde.icbc.PeriodicBC(geom=geometry_rectangle, component_x=0,
    on_boundary=periodic_boundary, derivative_order=1, component=0)

data_observ_points_LGM, observe_u_LGM = training_points(df_simulated_LGM)

model_lgm_constD, params_lgm_constD, best_step_lgm_constD = train_process_constD(
    data_observ_points_LGM,
    observe_u_LGM,
    D_const, bc_1, bc_2,
    model_name_lgm_constD,
    restore_path=holocene_constD_restore_path,
    epochs=150000, lr=1e-5
)

calculate_save_df_constD(
    model_lgm_constD,
    df_LGM_full_test.copy(),
    norm_params_LGM,
    RESULTS_PATH,
    "df_pinn_simulated_LGM_constD.csv"
)